In [ ]:
# Foodwise Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device
#     2. Daily Brand Page (LMS) Pageview/ Impressions
#     3. Daily Theme Listing Impressions 
#     4. Daily POIs Impressions (SR1 and or SR2)
#     5. Daily Map pin impressions

In [9]:
import datetime 
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}
from google.cloud import bigquery
client = bigquery.Client()

In [10]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

all_dates = pd.date_range(
    start=f"{year}-{str_month}-01",
    end=pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    freq="D",
)

In [ ]:
template_file = 'Foodwise_report_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

#excel_name = "Foodwise_report_2026-09-25.xlsx"

'../LSLS/report_for_BD_2026-09-28.xlsx'

# 咪嘥野食店 SR1 Icon Impression

In [ ]:
# sql = f'''
# with temp1 as 
#   ((select time, platform, split(replace((select item.value from unnest(eventlabel.list) where lower(item.param) = 'promotionid'), ' ', ''), ',') as promotionid from `openrice-production.ORGA.PV_{year}{str_month}*` 
#   where eventcategory = 'Promotion Related'
#   and (select item.value from unnest(eventlabel.list) where lower(item.param) = 'promotionid') like '%11%')
#   union all
#   (select time, platform, (select ARRAY_AGG(replace(regexp_extract(replace(lower(item.value), ' ', ''), 'promotionid:.*'), 'promotionid:', '')) as promotionid from unnest(eventlabel.list) where lower(item.param) = 'label')
#   from `openrice-production.ORGA.PV_{year}{str_month}*` 
#   where eventcategory = 'WebEvent'
#   and lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'label' and lower(item.value) like '%promotionid%' limit 1)) like '%promotionid%')),
# temp2 as 
#   (select '$$$咪嘥野食店$$$' AS dummy, time, platform from
#     (select time, platform, impression from temp1 cross join unnest(promotionid) as impression)
#   where impression = '11')

# select platform, date(time) as querydate, count(1) from temp2
# group by platform, querydate
# order by platform, querydate
#     '''

# df_big_query = client.query(sql).result().to_dataframe()

# pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='f0_')
# pivoted_df = pivoted_df.fillna(0)
# pivoted_df["Web"]=pivoted_df["desktop"]
# pivoted_df["Mobile Web"]=pivoted_df["mobile"]
# pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
# pivoted_df["IOS"]=pivoted_df["ios"]
# pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

# pivoted_df.index.name = None
# pivoted_df.reset_index(inplace=True)

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df.to_excel(writer, sheet_name='foodwise eateries', startrow=2, header=None, index=False)